In [24]:
%display latex
M = Manifold(4, 'M') #Variedad diferenciable 4-dimensional
X.<t, r, th, ph> = M.chart(r't r th:\theta ph:\phi') #Carta en coordenadas de Boyer-Lindquist
m, a = var('m a') #Constantes (masa y velocidad angular)

#Variables agrupadas
rho2 = r^2 + (a*cos(th))^2
Delta = r^2 - 2*m*r + a^2

#Definición de la métrica de Kerr
g = M.metric('g')
g[0,0] = -(1 - 2*m*r/rho2)
g[0,3] = -2*a*m*r*sin(th)^2 / rho2 
g[1,1] = rho2 / Delta
g[2,2] = rho2
g[3,3] = (r^2 + a^2 + 2*m*r*(a*sin(th))^2 / rho2) * sin(th)^2

g.display()

g = (2*m*r/(a^2*cos(th)^2 + r^2) - 1) dt*dt - 2*a*m*r*sin(th)^2/(a^2*cos(th)^2 + r^2) dt*dph + (a^2*cos(th)^2 + r^2)/(a^2 - 2*m*r + r^2) dr*dr + (a^2*cos(th)^2 + r^2) dth*dth - 2*a*m*r*sin(th)^2/(a^2*cos(th)^2 + r^2) dph*dt + (2*a^2*m*r*sin(th)^2/(a^2*cos(th)^2 + r^2) + a^2 + r^2)*sin(th)^2 dph*dph

In [25]:
## Parámetros exactos
caso = {m: 6, a: 5}
g.apply_map(lambda expr: expr.subs(caso)) #Sustitución de constantes en la métrica
#Horizontes
raiz_h = sqrt(m^2 - a^2)
raiz_ergo = sqrt(m^2 - a^2*cos(th)^2)
rh, rh2 = m + raiz_h, m - raiz_h
rergo, rergo2 = m + raiz_ergo, m - raiz_ergo
## Sustitución numérica en bloque
rhnum, rhnum2, rergonum, rergonum2 = [expr.subs(caso) for expr in (rh, rh2, rergo, rergo2)]

In [26]:
## Representación gráfica (estructura del agujero negro)

#Entorno de la representación
E3 = Manifold(3, 'E3') #Espacio euclídeo 3-dimensional.
Cart3.<x, y, z> = E3.chart(r'x y z') #Sistema de coordenadas cartesianas
Phi = M.diff_map(E3, {(X, Cart3): [r*sin(th)*cos(ph), r*sin(th)*sin(ph), r*cos(th)]}, name='Phi') #Carta de transformación a coordenadas cartesianas

plot_opts = {'axes_labels': ['', '', '']}
to_cartesian = lambda r_val: (r_val * sin(th) * cos(ph), r_val * sin(th) * sin(ph), r_val * cos(th))

#Horizontes de sucesos y ergosferas
grafico_horizonte = sphere((0,0,0), rhnum, color='grey', opacity=0.4, **plot_opts)
grafico_horizonte2 = sphere((0,0,0), rhnum2, color='black', opacity=0.6, **plot_opts)
grafico_ergosfera = parametric_plot3d(to_cartesian(rergonum), (th, 0, pi), (ph, 0, 2*pi), color='blue', opacity=0.2, **plot_opts)
grafico_ergosfera2 = parametric_plot3d(to_cartesian(rergonum2), (th, 0, pi), (ph, 0, 2*pi), color='purple', opacity=0.8, **plot_opts)

#Singularidad
radionum=a.subs(caso)
x_sing, y_sing, z_sing= radionum * sin(ph), -radionum * cos(ph), 0
grafico_singularidad= parametric_plot3d((x_sing, y_sing, z_sing), (ph, 0, 2*pi), color='green', opacity=1, **plot_opts)

#Dibujar todo
(grafico_horizonte + grafico_horizonte2 + grafico_ergosfera + grafico_ergosfera2 + grafico_singularidad).show(frame=False)

Graphics3d Object

In [48]:
#Función que nos permite conocer el vector temporal o luz conocida la dirección espacial y la norma (-m²) del mismo
def calc_temp(M, g, pos_inicial, vect_espacial, m):
    x = var('x')
    p = M.point(pos_inicial, name='p')
    Tp = M.tangent_space(p)
    frame = M.default_chart().frame().at(p)
    v_total = [x] + list(vect_espacial)
    v = Tp(v_total, basis=frame)
    norm = g.at(p)(v, v)
    eq = norm == -m^2
    sol = solve(eq, x)
    chosen = sol[0] if sol[0]>=0 else sol[-1]
    vfinal = [chosen.rhs()] + list(vect_espacial)
    v_final = Tp(vfinal, basis=frame)
    return v_final

In [49]:
#Función que calcula el radio y la velocidad axial para las cuales se tienen órbitas circulares (si existen) dadas un th inicial
#y una velocidad en th nula

def calc_orbita(M, g, theta=pi/2, m=0):
    g00, g03, g22, g33 = [g[i,j].expr().subs({th: theta}).subs(caso) for i, j in [(0,0), (0,3), (2,2), (3,3)]]
    
    v_ph=var('v_ph')
    
    eq1 = g00 + g33*v_ph^2 + 2*g03*v_ph==-m^2
    eq2 = diff(g00,r) + diff(g33,r)*v_ph^2 + 2*diff(g03,r)*v_ph==0
    soluciones_orbita = [sol for sol in solve([eq1, eq2], (r,v_ph)) if sol[0].rhs() > rhnum] 
    r1 = N(soluciones_orbita[0][0].rhs(), digits=25)
    r2 = N(soluciones_orbita[1][0].rhs(), digits=25)
    v_ph1 = N(soluciones_orbita[0][1].rhs(), digits=25)
    v_ph2 = N(soluciones_orbita[1][1].rhs(), digits=25)
    orbita_circular = [r1, r2, v_ph1, v_ph2]
    
    return orbita_circular

#Se buscan posibles órbitas circulares sustituyendo las soluciones anteriores como parámetros de entrada al calcular geodésicas

In [50]:
## Agujero negro (se emplea en el resto de representaciones)

grafico_horizonte = sphere((0,0,0), rhnum, color='black', opacity=0.8)
grafico_ergosfera = parametric_plot3d(to_cartesian(rergonum), (th, 0, pi), (ph, 0, 2*pi), color='blue', opacity=0.4)

## Función de cálculo y representación de geodésicas

def crear_geodesica(M, g, Phi, pos_inicial, vect_espacial, s_max, nombre="geo", 
                    color_linea='red', grosor=2, rtol=1e-10, atol=1e-10,
                    dibujar_punto=True, dibujar_vector=True, escala_vector=15.0, 
                    anchura_vector=3, tamaño_punto=20, norma=0):
    
    #Cálculo del vector inicial
    vfull = calc_temp(M, g, pos_inicial, vect_espacial, norma)
    p = vfull.parent().base_point()
    if vfull[0]<0:
        print(f"La geodésica {nombre} va hacia el pasado")
    
    
    #Cálculo de la geodésica
    s = var(f's_{nombre}') 
    c = M.integrated_geodesic(g, (s, 0, s_max), vfull, name=f'c_{nombre}') 
    c.solve(rtol=rtol, atol=atol)
    c.interpolate() 
      
    #Gráfico de la geodésica
    grafico_resultado = c.plot_integrated(
        mapping=Phi, 
        thickness=grosor, 
        color=color_linea, 
        label_axes=False 
    )

    #Punto inicial y vector de velocidad inicial
    if dibujar_punto or dibujar_vector:
        coords_3d = Phi(p).coord()
        x_val, y_val, z_val = coords_3d[0], coords_3d[1], coords_3d[2]
          
        if dibujar_punto:
            grafico_resultado += point3d(
                (x_val, y_val, z_val), 
                size=tamaño_punto, 
                color='orange', 
                frame=False
            )
              
        if dibujar_vector:
            v_3d = Phi.differential(p)(vfull)
            vx, vy, vz = v_3d[0], v_3d[1], v_3d[2]
            grafico_resultado += arrow3d(
                (x_val, y_val, z_val), 
                (x_val + vx * escala_vector, y_val + vy * escala_vector, z_val + vz * escala_vector), 
                color=color_linea, 
                width=anchura_vector, 
                frame=False
            )
              
    return grafico_resultado

In [51]:
## Representación gráfica (geodésicas temporales escapando y cayendo)

grafico_geodesica1 = crear_geodesica(M, g, Phi, 
                             (0,14.0,RR(pi/2),0), 
                             (0.0, 0.06, 0.13), 
                             s_max=60, nombre="1", color_linea='red', 
                             dibujar_punto=True, dibujar_vector=True, escala_vector=3.0,
                             anchura_vector=4, tamaño_punto=160, norma=-1)

grafico_geodesica2 = crear_geodesica(M, g, Phi, 
                             (0,14.0,RR(pi/2),0), 
                             (0.0, 0.005, 0.028), 
                             s_max=15.6, nombre="2", color_linea='green', 
                             dibujar_punto=True, dibujar_vector=True, escala_vector=15.0,
                             anchura_vector=4, tamaño_punto=160, norma=-1)

(grafico_horizonte + grafico_ergosfera + grafico_geodesica1 + grafico_geodesica2).show(frame=False)

Graphics3d Object

In [52]:
## Representación gráfica (geodésicas orbitando)

#Geodésica circular
grafico_geodesica1 = crear_geodesica(M, g, Phi, 
                             (0,25.0,RR(pi/2),0), 
                             (0.0, 0.0, 0.0284040820577346), 
                             s_max=250, nombre="1", color_linea='green', 
                             dibujar_punto=True, dibujar_vector=True, escala_vector=20.0,
                             anchura_vector=15, tamaño_punto=350, norma=-1)

#Geodésica elíptica (precesión relativista)
grafico_geodesica2 = crear_geodesica(M, g, Phi, 
                             (0,50.0,RR(pi/2),0), 
                             (0.0, 0.0, 0.007403782447226678), 
                             s_max=2000, nombre="2", color_linea='red', 
                             dibujar_punto=True, dibujar_vector=True, escala_vector=40.0,
                             anchura_vector=15, tamaño_punto=350, norma=-1)

(grafico_horizonte + grafico_ergosfera + grafico_geodesica1 + grafico_geodesica2).show(frame=False)

Graphics3d Object

In [53]:
## Representación gráfica (frame dragging)

#Geodésica con velocidad inicial espacialmente nula
grafico_geodesica1 = crear_geodesica(M, g, Phi, 
                             (0,12.3,RR(pi/2),0), 
                             (0.0, 0.0, 0.0), 
                             s_max=3.8, nombre="1", color_linea='red', 
                             dibujar_punto=True, dibujar_vector=True, escala_vector=20.0,
                             anchura_vector=15, tamaño_punto=200, norma=-1)

#Geodésica con velocidad inicial en contra del giro del agujero negro
grafico_geodesica2 = crear_geodesica(M, g, Phi, 
                             (0,11.7,RR(pi/3),0), 
                             (0.0, 0.0, -1), 
                             s_max=0.0599, nombre="2", color_linea='green', 
                             dibujar_punto=True, dibujar_vector=True, escala_vector=0.5,
                             anchura_vector=3, tamaño_punto=200, norma=-1/100)

(grafico_horizonte + grafico_ergosfera + grafico_geodesica1 + grafico_geodesica2).show(frame=False)

Graphics3d Object

In [54]:
## Representación gráfica (iteración de la velocidad inicial, siendo v_th=0)

#Bucle de geodésicas a favor del giro
lista_graficos = []
for i in range(0, 8, 1):
    grafico_geodesica_i = crear_geodesica(M, g, Phi, 
                             (0,15,RR(pi/2),0), 
                             (0.0, 0.0, 0.025), 
                             s_max=90, nombre=str(i), color_linea='red', 
                             dibujar_punto=True, dibujar_vector=False, escala_vector=20.0,
                             anchura_vector=15, tamaño_punto=200, norma=i/20)
    lista_graficos.append(grafico_geodesica_i)

grafico_geodesica_aux = sum(lista_graficos)

#Bucle de geodésicas en contra del giro
lista_graficos2 = []
for i in range(0, 8, 2):
    grafico_geodesica_i = crear_geodesica(M, g, Phi, 
                             (0,15,RR(pi/2),0), 
                             (0.0, 0.0, -0.025), 
                             s_max=16.7, nombre=str(i), color_linea='red', 
                             dibujar_punto=True, dibujar_vector=False, escala_vector=20.0,
                             anchura_vector=15, tamaño_punto=200, norma=i/20)
    lista_graficos2.append(grafico_geodesica_i)

grafico_geodesica_aux2 = sum(lista_graficos2)

(grafico_horizonte + grafico_ergosfera + grafico_geodesica_aux + grafico_geodesica_aux2).show(frame=False)

Graphics3d Object

In [55]:
## Representación gráfica (órbitas circulares de geodésicas luz)

parametros = calc_orbita(M, g, theta=pi/2, m=0)

#Anillo de luz 1 (retrógada)
grafico_geodesica1 = crear_geodesica(M, g, Phi, 
                             (0,parametros[0],RR(pi/2),0), 
                             (0.0, 0.0, parametros[2]), 
                             s_max=300, nombre="1", color_linea='green', rtol=1e-12, atol=1e-12,
                             dibujar_punto=True, dibujar_vector=True, escala_vector=10.0,
                             anchura_vector=4, tamaño_punto=200, norma=0)

#Anillo de luz 2 (prógrada)
grafico_geodesica2 = crear_geodesica(M, g, Phi, 
                             (0,parametros[1],RR(pi/2),0), 
                             (0.0, 0.0, parametros[3]), 
                             s_max=7.95, nombre="2", color_linea='red', rtol=1e-13, atol=1e-18,
                             dibujar_punto=True, dibujar_vector=True, escala_vector=10.0,
                             anchura_vector=4, tamaño_punto=200, norma=0)

(grafico_horizonte + grafico_ergosfera + grafico_geodesica1 + grafico_geodesica2).show(frame=False)

Graphics3d Object

In [56]:
## Representación gráfica (órbitas circulares de geodésicas luz en función del sentido de la velocidad inicial)

parametros = calc_orbita(M, g, theta=pi/2, m=0)

#Geodésica luz 1 (amillo luz)
grafico_geodesica1 = crear_geodesica(M, g, Phi, 
                             (0,parametros[0],RR(pi/2),0), 
                             (0.0, 0.0, parametros[2]), 
                             s_max=300, nombre="1", color_linea='red', rtol=1e-12, atol=1e-12,
                             dibujar_punto=True, dibujar_vector=True, escala_vector=10.0,
                             anchura_vector=3, tamaño_punto=200, norma=0)

#Geodésica luz 2 (escapa)
grafico_geodesica2 = crear_geodesica(M, g, Phi, 
                             (0,parametros[0],RR(pi/2),0), 
                             (0.0, 0.0, -parametros[2]), 
                             s_max=60, nombre="2", color_linea='green', rtol=1e-12, atol=1e-12,
                             dibujar_punto=True, dibujar_vector=True, escala_vector=10.0,
                             anchura_vector=3, tamaño_punto=200, norma=0)

(grafico_horizonte + grafico_ergosfera + grafico_geodesica1 + grafico_geodesica2).show(frame=False)

Graphics3d Object

In [57]:
#Representación gráfica (órbitas circulares de geodésicas temporales en función del sentido de la velocidad inicial)

#Geodésica temporal 1
grafico_geodesica1 = crear_geodesica(M, g, Phi, 
                             (0,25.0,RR(pi/2),0), 
                             (0.0, 0.0, 0.0284040820577346), 
                             s_max=250, nombre="1", color_linea='red', 
                             dibujar_punto=True, dibujar_vector=True, escala_vector=10.0,
                             anchura_vector=3, tamaño_punto=250, norma=-1)

#Geodésica temporal 2
grafico_geodesica2 = crear_geodesica(M, g, Phi, 
                             (0,25.0,RR(pi/2),0), 
                             (0.0, 0.0, -0.0284040820577346), 
                             s_max=40, nombre="2", color_linea='green', 
                             dibujar_punto=True, dibujar_vector=True, escala_vector=10.0,
                             anchura_vector=3, tamaño_punto=250, norma=-1)

(grafico_horizonte + grafico_ergosfera + grafico_geodesica1 + grafico_geodesica2).show(frame=False)

Graphics3d Object

In [58]:
#Representación gráfica (geodésicas luz con posición inicial fuera del ecuador)

#Geodésica 1
grafico_geodesica1 = crear_geodesica(M, g, Phi, 
                             (0,13.0,RR(pi/4),0), 
                             (0.0, 0.0, 1), 
                             s_max=15, nombre="1", color_linea='red', 
                             dibujar_punto=True, dibujar_vector=True, escala_vector=0.4,
                             anchura_vector=4, tamaño_punto=250, norma=0)

#Geodésica 2
grafico_geodesica2 = crear_geodesica(M, g, Phi, 
                             (0,15.0,RR(pi/4),0), 
                             (0.0, 0.0, 1), 
                             s_max=8, nombre="2", color_linea='green', 
                             dibujar_punto=True, dibujar_vector=True, escala_vector=0.35,
                             anchura_vector=4, tamaño_punto=250, norma=0)

#Geodésica 3
grafico_geodesica3 = crear_geodesica(M, g, Phi, 
                             (0,18.0,RR(pi/4),0), 
                             (0.0, 0.0, 1), 
                             s_max=3, nombre="3", color_linea='purple', 
                             dibujar_punto=True, dibujar_vector=True, escala_vector=0.27,
                             anchura_vector=4, tamaño_punto=250, norma=0)

#Geodésica 4
grafico_geodesica4 = crear_geodesica(M, g, Phi, 
                             (0,11.0,RR(pi/4),0), 
                             (0.0, 0.0, 1), 
                             s_max=4.5, nombre="4", color_linea='brown', 
                             dibujar_punto=True, dibujar_vector=True, escala_vector=0.45,
                             anchura_vector=4, tamaño_punto=250, norma=0)

(grafico_horizonte + grafico_ergosfera + grafico_geodesica1 + grafico_geodesica2 + grafico_geodesica3 + grafico_geodesica4).show(frame=False)

Graphics3d Object

In [59]:
#Representación gráfica (geodésicas luz con posición inicial fuera del ecuador)

#Geodésica 1
grafico_geodesica1 = crear_geodesica(M, g, Phi, 
                             (0,13.0,RR(pi/4),0), 
                             (0.0, 0.0, -1), 
                             s_max=0.6, nombre="1", color_linea='red', 
                             dibujar_punto=True, dibujar_vector=True, escala_vector=0.4,
                             anchura_vector=4, tamaño_punto=250, norma=0)

#Geodésica 2
grafico_geodesica2 = crear_geodesica(M, g, Phi, 
                             (0,15.0,RR(pi/4),0), 
                             (0.0, 0.0, -1), 
                             s_max=1.05, nombre="2", color_linea='green', 
                             dibujar_punto=True, dibujar_vector=True, escala_vector=0.35,
                             anchura_vector=4, tamaño_punto=250, norma=0)

#Geodésica 3
grafico_geodesica3 = crear_geodesica(M, g, Phi, 
                             (0,18.0,RR(pi/4),0), 
                             (0.0, 0.0, -1), 
                             s_max=1.9, nombre="3", color_linea='purple', 
                             dibujar_punto=True, dibujar_vector=True, escala_vector=0.27,
                             anchura_vector=4, tamaño_punto=250, norma=0)

#Geodésica 4
grafico_geodesica4 = crear_geodesica(M, g, Phi, 
                             (0,11.0,RR(pi/4),0), 
                             (0.0, 0.0, -1), 
                             s_max=0.055, nombre="4", color_linea='brown', 
                             dibujar_punto=True, dibujar_vector=True, escala_vector=0.45,
                             anchura_vector=4, tamaño_punto=250, norma=0)

(grafico_horizonte + grafico_ergosfera + grafico_geodesica1 + grafico_geodesica2 + grafico_geodesica3 + grafico_geodesica4).show(frame=False)

Graphics3d Object

In [60]:
#Representación gráfica (búsqueda de órbitas luz circulares con posición inicial fuera del ecuador)

parametros = calc_orbita(M, g, theta=pi/4, m=0)

#órbita 1
grafico_geodesica1 = crear_geodesica(M, g, Phi, 
                             (0,parametros[0],RR(pi/4),0), 
                             (0.0, 0.0, parametros[2]), 
                             s_max=450, nombre="1", color_linea='red', 
                             dibujar_punto=True, dibujar_vector=True, escala_vector=10,
                             anchura_vector=4, tamaño_punto=250, norma=0)

#órbita 2
grafico_geodesica2 = crear_geodesica(M, g, Phi, 
                             (0,parametros[1],RR(pi/4),0), 
                             (0.0, 0.0, parametros[3]), 
                             s_max=450, nombre="2", color_linea='green', 
                             dibujar_punto=True, dibujar_vector=True, escala_vector=10,
                             anchura_vector=4, tamaño_punto=250, norma=0)

(grafico_horizonte + grafico_ergosfera + grafico_geodesica1 + grafico_geodesica2).show(frame=False)

Graphics3d Object

In [61]:
#Representación gráfica (geodésicas luz con posición inicial y velocidad inicial fuera del ecuador)

#Geodésica 1
grafico_geodesica1 = crear_geodesica(M, g, Phi, 
                             (0,13.0,RR(pi/4),0), 
                             (0.0, 1, 0.0), 
                             s_max=0.95, nombre="1", color_linea='red', 
                             dibujar_punto=True, dibujar_vector=True, escala_vector=0.4,
                             anchura_vector=4, tamaño_punto=250, norma=0)

#Geodésica 2
grafico_geodesica2 = crear_geodesica(M, g, Phi, 
                             (0,15.0,RR(pi/4),0), 
                             (0.0, 1, 0.0), 
                             s_max=1.6, nombre="2", color_linea='green', 
                             dibujar_punto=True, dibujar_vector=True, escala_vector=0.35,
                             anchura_vector=4, tamaño_punto=250, norma=0)

#Geodésica 3
grafico_geodesica3 = crear_geodesica(M, g, Phi, 
                             (0,18.0,RR(pi/4),0), 
                             (0.0, 1, 0.0), 
                             s_max=3.2, nombre="3", color_linea='purple', 
                             dibujar_punto=True, dibujar_vector=True, escala_vector=0.27,
                             anchura_vector=4, tamaño_punto=250, norma=0)

#Geodésica 4
grafico_geodesica4 = crear_geodesica(M, g, Phi, 
                             (0,20.0,RR(pi/4),0), 
                             (0.0, 1, 0.0), 
                             s_max=5.3, nombre="4", color_linea='brown', 
                             dibujar_punto=True, dibujar_vector=True, escala_vector=0.25,
                             anchura_vector=4, tamaño_punto=250, norma=0)

(grafico_horizonte + grafico_ergosfera + grafico_geodesica1 + grafico_geodesica2 + grafico_geodesica3 + grafico_geodesica4).show(frame=False)

Graphics3d Object

In [62]:
#Función que calcula el radio y la velocidad axial para las cuales se tienen órbitas circulares (si existen) dadas un th inicial
#y una velocidad en th nula

def calc_orbita2(M, g, theta=pi/2, m=0):
    
    g00 = g[0,0].expr().subs({th: theta}).subs(caso)
    g22 = g[2,2].expr().subs({th: theta}).subs(caso)
    g33 = g[3,3].expr().subs({th: theta}).subs(caso)
    
    v_th=var('v_th')
    
    eq1 = g00 + g22*v_th^2 ==-m^2
    eq2 = diff(g00,r) + diff(g22,r)*v_th^2 ==0
    soluciones = solve([eq1, eq2], (r,v_th))
    soluciones_orbita = [sol for sol in soluciones if sol[0].rhs() > rhnum]
    
    r1 = N(soluciones_orbita[0][0].rhs(), digits=25)
    r2 = N(soluciones_orbita[1][0].rhs(), digits=25)
    v_th1 = N(soluciones_orbita[0][1].rhs(), digits=25)
    v_th2 = N(soluciones_orbita[1][1].rhs(), digits=25)
    orbita_circular = [r1, r2, v_th1, v_th2]
    
    return orbita_circular

#Se buscan posibles órbitas circulares sustituyendo las soluciones anteriores como parámetros de entrada al calcular geodésicas

In [63]:
#Representación gráfica (búsqueda de órbitas luz con posición inicial y velocidad inicial fuera del ecuador)

parametros2= calc_orbita2(M, g, theta=pi/4, m=0)

#Geodésica 1
grafico_geodesica1 = crear_geodesica(M, g, Phi, 
                             (0,parametros2[0],RR(pi/4),0), 
                             (0.0, parametros2[2], 0.0), 
                             s_max=300, nombre="1", color_linea='red', 
                             dibujar_punto=True, dibujar_vector=True, escala_vector=10,
                             anchura_vector=4, tamaño_punto=250, norma=0)

#Geodésica 2
grafico_geodesica2 = crear_geodesica(M, g, Phi, 
                             (0,parametros2[1],RR(pi/4),0), 
                             (0.0, parametros2[3], 0.0), 
                             s_max=300, nombre="2", color_linea='green', 
                             dibujar_punto=True, dibujar_vector=True, escala_vector=10,
                             anchura_vector=4, tamaño_punto=250, norma=0)

(grafico_horizonte + grafico_ergosfera + grafico_geodesica1 + grafico_geodesica2).show(frame=False)

Graphics3d Object

In [64]:
#Representación gráfica (búsqueda de órbitas luz con velocidad inicial fuera del ecuador)

parametros2= calc_orbita2(M, g, theta=pi/2, m=0)

#Geodésica 1
grafico_geodesica1 = crear_geodesica(M, g, Phi, 
                             (0,parametros2[0],RR(pi/2),0), 
                             (0.0, parametros2[2], 0.0), 
                             s_max=300, nombre="1", color_linea='red', 
                             dibujar_punto=True, dibujar_vector=True, escala_vector=10,
                             anchura_vector=4, tamaño_punto=250, norma=0)

#Geodésica 2
grafico_geodesica2 = crear_geodesica(M, g, Phi, 
                             (0,parametros2[1],RR(pi/2),0), 
                             (0.0, parametros2[3], 0.0), 
                             s_max=300, nombre="2", color_linea='green', 
                             dibujar_punto=True, dibujar_vector=True, escala_vector=10,
                             anchura_vector=4, tamaño_punto=250, norma=0)

(grafico_horizonte + grafico_ergosfera + grafico_geodesica1 + grafico_geodesica2).show(frame=False)

Graphics3d Object